#Silver Layer

###Studies

In [0]:
%sql
CREATE OR REPLACE TABLE silver.studies_diabetes AS
WITH base AS (
    SELECT DISTINCT
        s.nct_id AS study_id,
        s.brief_title,
        COALESCE(
            CASE WHEN s.start_date_type = 'ACTUAL' THEN s.start_date else null END,
            s.start_date
        ) AS start_date,
        COALESCE(
            CASE WHEN s.completion_date_type = 'ACTUAL' THEN s.completion_date else null END,
            s.completion_date
        ) AS completion_date,
        s.study_type,
        CASE
    WHEN upper(phase) LIKE '%PHASE%1%/%PHASE%2%' THEN 'PHASE1'
    WHEN upper(phase) LIKE '%PHASE%2%/%PHASE%3%' THEN 'PHASE3'
    WHEN upper(phase) LIKE '%EARLY%' THEN 'PHASE1'
    WHEN upper(phase) LIKE '%PHASE 1%' THEN 'PHASE1'
    WHEN upper(phase) LIKE '%PHASE 2%' THEN 'PHASE2'
    WHEN upper(phase) LIKE '%PHASE 3%' THEN 'PHASE3'
    WHEN upper(phase) LIKE '%PHASE 4%' THEN 'PHASE4'
    WHEN upper(phase) =('NA') THEN 'NOT REPORTED'
    WHEN upper(phase) IS NULL THEN 'NOT REPORTED'

    ELSE phase
END AS clinical_phase,
        CASE 
            WHEN s.overall_status LIKE '%UNKNOWN%' THEN NULL 
            ELSE s.overall_status 
        END AS overall_status
    FROM bronze.raw_studies s
    JOIN bronze.raw_conditions c 
        ON s.nct_id = c.nct_id   
    WHERE
        LOWER(c.name) LIKE '%diabetes%'
        AND s.start_date > '2000-01-01'
        AND s.study_type = 'INTERVENTIONAL'
        AND s.official_title IS NOT NULL
)
SELECT
    study_id,
    brief_title,
    start_date,
    completion_date,
    study_type,
    clinical_phase,
    overall_status,
    
    CASE
        WHEN trim(upper(overall_status)) = 'COMPLETED'
             AND regexp_extract(trim(upper(clinical_phase)), '([0-9])', 1) IN ('3','4')
        THEN 1
        WHEN trim(upper(overall_status)) IN ('TERMINATED', 'WITHDRAWN', 'SUSPENDED')
        THEN 0
        ELSE NULL
    END AS approved
FROM base;
select * from silver.studies_diabetes limit 10


study_id,brief_title,start_date,completion_date,study_type,clinical_phase,overall_status,approved
NCT04604223,Effect of Pioglitazone on T2DM Patients With COVID-19,2021-01-18,2021-06-29,INTERVENTIONAL,PHASE4,null,null
NCT06117631,Project Sueño: Sleep & Understanding Early Nutrition in Obesity,2023-07-01,2027-12-31,INTERVENTIONAL,NOT REPORTED,ACTIVE_NOT_RECRUITING,null
NCT02923960,Effect of Nutritional Products in Subjects With Type 2 Diabetes,2016-10-31,2017-01-31,INTERVENTIONAL,NOT REPORTED,COMPLETED,null
NCT02589756,Fish or Nuts? Dietary Effects on Cardiometabolic Risk Factors and Persistent Organic Pollutants,2015-09-30,2017-09-11,INTERVENTIONAL,NOT REPORTED,COMPLETED,null
NCT04965935,"Efficacy, Mechanisms and Safety of SGLT2 Inhibitors in Kidney Transplant Recipients",2021-07-15,2024-08-30,INTERVENTIONAL,PHASE3,COMPLETED,1
NCT02605889,Acupuncture in the Complementary Treatment of Diabetes Mellitus Type II,2014-10-31,2016-01-31,INTERVENTIONAL,NOT REPORTED,null,null
NCT04997512,Freestyle Libre and Hospital Admissions in Type 2 Diabetes,2021-11-01,2025-08-01,INTERVENTIONAL,NOT REPORTED,null,null
NCT00749918,Effect of Vitamin D Deficiency and Vitamin D Supplementation on Glucose Metabolism,2007-01-31,2008-06-30,INTERVENTIONAL,NOT REPORTED,COMPLETED,null
NCT01792986,The Fasting II Study,2013-02-28,2013-06-30,INTERVENTIONAL,NOT REPORTED,COMPLETED,null
NCT02420392,Effects of Dapagliflozin on the Incretin Sensitivity of the Pancreatic Beta Cell,2015-02-28,2016-04-30,INTERVENTIONAL,NOT REPORTED,COMPLETED,null


###Conditions

In [0]:
%sql
CREATE OR REPLACE TABLE silver.conditions AS
SELECT
    c.nct_id AS study_id,
    c.id AS condition_id,
    CASE 
        WHEN LOWER(c.name) IN ('na','n/a','','none','not provided') THEN NULL
        ELSE c.name
    END AS condition_name_raw,
    CASE
        WHEN LOWER(c.name) LIKE '%type 1 diabetes%' 
             OR LOWER(c.name) LIKE '%type i diabetes%' 
             OR LOWER(c.name) LIKE '%juvenile%' 
            THEN 'Type 1 Diabetes'

        WHEN LOWER(c.name) LIKE '%type 2 diabetes%' 
             OR LOWER(c.name) LIKE '%type ii diabetes%' 
             OR LOWER(c.name) LIKE '%t2dm%' 
            THEN 'Type 2 Diabetes'

        WHEN LOWER(c.name) LIKE '%gestational%' 
             OR LOWER(c.name) LIKE '%pregnancy%' 
            THEN 'Gestational Diabetes'

        WHEN LOWER(c.name) LIKE '%prediabetes%' 
             OR LOWER(c.name) LIKE '%impaired glucose%' 
            THEN 'Prediabetes'

        WHEN LOWER(c.name) LIKE '%mody%' 
             OR LOWER(c.name) LIKE '%monogenic%' 
            THEN 'Monogenic Diabetes'

        WHEN LOWER(c.name) LIKE '%diabetes%' 
             OR LOWER(c.name) LIKE '%hyperglycemia%' 
            THEN 'Diabetes Mellitus'

        ELSE 'Non-Diabetes'
    END AS condition_category
FROM bronze.raw_conditions c;
select * from silver.conditions
limit 10

study_id,condition_id,condition_name_raw,condition_category
NCT01344096,380504840,Gaucher Disease,Non-Diabetes
NCT01344096,380504841,Thrombocytopathy,Non-Diabetes
NCT04846725,380504842,Venous Thromboembolic Disease,Non-Diabetes
NCT01627990,380504843,Solid Tumour,Non-Diabetes
NCT01627990,380504844,Malignant Haematological Tumour,Non-Diabetes
NCT01627990,380504845,Primary or Secondary Prophylactic Treatment,Non-Diabetes
NCT02963493,380504846,Multiple Myeloma,Non-Diabetes
NCT00337103,380504847,Metastatic Breast Cancer,Non-Diabetes
NCT06218407,380504848,Chronic Pain,Non-Diabetes
NCT06218407,380504849,Low Back Pain,Non-Diabetes


###Countries 

In [0]:
%sql
CREATE OR REPLACE TABLE silver.countries AS
SELECT
    con.id AS country_id,              
    con.nct_id AS study_id,             
    CASE
        WHEN con.name ILIKE '%czech%'     THEN 'Czech Republic'
        WHEN con.name ILIKE '%guinea%'    THEN 'Guinea'
        WHEN con.name ILIKE '%monaco%'    THEN 'France'
        WHEN con.name ILIKE '%réunion%'   THEN 'France'
        WHEN con.name ILIKE '%serbia%'   THEN 'Serbia and Montenegro'

        ELSE con.name
    END AS country_name
    FROM bronze.raw_countries con
WHERE
    con.removed = 'false'
    AND con.name IS NOT NULL;
    select *
     from silver.countries limit 10


country_id,study_id,country_name
296230292,NCT05252052,United States
296230293,NCT01036763,Germany
296230294,NCT00743275,United States
296230295,NCT00575029,United States
296230296,NCT00448409,United States
296230297,NCT01341756,Singapore
296230298,NCT02972463,Canada
296230299,NCT01980654,United States
296230300,NCT00873353,Spain
296230301,NCT06280326,Turkey (Türkiye)


###Interventions

In [0]:
%sql
CREATE OR REPLACE TABLE silver.interventions AS
SELECT
    i.id        AS intervention_id,
    i.nct_id    AS study_id,
    CASE 
        WHEN i.name IN ('NA', 'N/A', '', 'None', 'Not Provided') THEN NULL
        ELSE i.name
    END         AS intervention_name,
    i.intervention_type
FROM bronze.raw_interventions i
WHERE
    i.name IS NOT NULL
  AND NOT (LOWER(i.name) IN ('na','n/a','','none','not provided'));
    select * from silver.interventions limit 10


intervention_id,study_id,intervention_name,intervention_type
365831955,NCT00574717,Spectacles,PROCEDURE
365431066,NCT04524663,Camostat Mesilate,DRUG
365831956,NCT05968768,Naxitamab,DRUG
365831957,NCT05924347,MRI,DIAGNOSTIC_TEST
365431067,NCT04524663,Placebo,DRUG
365831958,NCT05924347,Scolioscan,DIAGNOSTIC_TEST
365831959,NCT05924347,Skeletal maturity assessment,DIAGNOSTIC_TEST
365831960,NCT01946152,Dexamethasone,DRUG
365431068,NCT04524663,Standard of Care Treatment,OTHER
365437352,NCT03629483,Dexmedetomidine,DRUG


###Sponsors

In [0]:
%sql
CREATE OR REPLACE TABLE silver.sponsors AS
SELECT
    sp.id                 AS sponsor_id,
    sp.nct_id             AS study_id,
    CASE 
        WHEN lower(sp.name) IN ('na', 'n/a', '', 'none', 'not provided') THEN NULL
        ELSE sp.name
    END                   AS sponsor_name,
    sp.agency_class,
    sp.lead_or_collaborator
FROM bronze.raw_sponsors sp
WHERE
    sp.name IS NOT NULL;
    select * from silver.sponsors limit 10


sponsor_id,study_id,sponsor_name,agency_class,lead_or_collaborator
352221349,NCT01437345,"FSH Society, Inc.",INDUSTRY,collaborator
352221350,NCT01437345,FSHD Global Research Foundation,OTHER,collaborator
352221351,NCT01437345,Muscular Dystrophy Canada,OTHER,collaborator
352221352,NCT01437345,"aTyr Pharma, Inc.",INDUSTRY,collaborator
352221353,NCT06210269,Tampere University Hospital,OTHER,collaborator
352221354,NCT06210269,Oulu University Hospital,OTHER,collaborator
352221355,NCT06210269,Kuopio University Hospital,OTHER,collaborator
352221356,NCT06210269,Pori Central Hospital,UNKNOWN,collaborator
352221357,NCT06210269,Jyväskylä Central Hospital,OTHER,collaborator
352221358,NCT06210269,Mikkeli Central Hospital,OTHER,collaborator


###Designs

In [0]:
%sql
CREATE OR REPLACE TABLE silver.designs AS
SELECT DISTINCT
    d.id                  AS design_id,
    d.nct_id              AS study_id,
    case when d.allocation like '%NA%' then null else d.allocation end as allocation,
    d.primary_purpose,
    d.intervention_model,
    d.masking,
    d.subject_masked,
    d.caregiver_masked,
    d.investigator_masked
FROM bronze.raw_designs d;
select * from silver.designs limit 10


design_id,study_id,allocation,primary_purpose,intervention_model,masking,subject_masked,caregiver_masked,investigator_masked
214083123,NCT03059316,RANDOMIZED,SUPPORTIVE_CARE,SEQUENTIAL,SINGLE,false,false,true
214083151,NCT04775953,RANDOMIZED,TREATMENT,PARALLEL,NONE,null,null,null
214083153,NCT03211494,RANDOMIZED,PREVENTION,CROSSOVER,NONE,null,null,null
214083161,NCT06647472,null,null,null,null,null,null,null
214083165,NCT01002651,RANDOMIZED,SUPPORTIVE_CARE,CROSSOVER,QUADRUPLE,true,true,true
214083172,NCT02759250,NON_RANDOMIZED,TREATMENT,PARALLEL,NONE,null,null,null
214083229,NCT04420975,null,TREATMENT,SINGLE_GROUP,NONE,null,null,null
213965095,NCT03410212,RANDOMIZED,TREATMENT,PARALLEL,QUADRUPLE,true,true,true
213965098,NCT05024981,RANDOMIZED,TREATMENT,PARALLEL,SINGLE,true,false,false
213610777,NCT00617292,null,null,null,null,null,null,null


###Facilities

In [0]:
%sql
CREATE OR REPLACE TABLE silver.facilities AS
SELECT DISTINCT
    f.id        AS facility_id,
    f.nct_id    AS study_id,
    f.name      AS facility_name,
    f.city,
    f.state,
    f.country,
    f.latitude,
    f.longitude
FROM bronze.raw_facilities f
WHERE
    f.name IS NOT NULL;
    select * from silver.facilities limit 10


facility_id,study_id,facility_name,city,state,country,latitude,longitude
1311814760,NCT05407168,Hillsboro Medical Center,Hillsboro,Oregon,United States,45.522890,-122.989830
1313213368,NCT02354586,GSK Investigational Site,Nashville,Tennessee,United States,36.165890,-86.784440
1313213446,NCT05911139,Department of paediatric anaesthesiology and intensive medicine,Bratislava,null,Slovakia,48.148160,17.106740
1313213460,NCT00728949,ImClone Investigational Site,Westwood,Kansas,United States,39.040560,-94.616900
1313213514,NCT00118105,West Michigan Cancer Center,Kalamazoo,Michigan,United States,42.291710,-85.587230
1313213530,NCT00118105,Kalispell Regional Medical Center,Kalispell,Montana,United States,48.195790,-114.312910
1313213584,NCT01424722,Watson Clinic Center,Lakeland,Florida,United States,28.039470,-81.949800
1313213630,NCT01424722,Oklahoma Heart Institute at Utica,Tulsa,Oklahoma,United States,36.153980,-95.992770
1313213651,NCT01424722,The Hope Heart Institute,Bellevue,Washington,United States,47.610380,-122.200680
1313213739,NCT00084084,East Tennessee Children's Hospital,Knoxville,Tennessee,United States,35.960640,-83.920740
